## Tutorial
$$\text{d} s_{t} = g(s_{t}) d t \, - h(s_{t^{-}}) d \mathcal{P}_t(\lambda)$$
In this tutorial, we show how to use our Bayesian inference framework to learn the above stochastic differential equation. Here, we take a simulated cell growth and division trajectory as input data. The code to simulate the trajectory is also provided below.

We start with the required packages for our Bayesian inference framework.

In [1]:
# include packages
using LinearAlgebra
using DifferentialEquations, Plots
using BenchmarkTools
using HypothesisTests
using DelimitedFiles
default(grid=false,legend=false, size=(400,320), dpi=300, tickfontsize=9, lw=2, lc=:gray, labelfontsize=9, tickdir=:out, fontfamily="Arial", legendfontsize=7, markercolor=:indianred, thickness_scaling=2)

mydir = "../helper/"
include(mydir * "base.jl")
include(mydir * "optimization.jl")
include(mydir * "misc.jl")
include(mydir * "myplot.jl")

plot_output (generic function with 3 methods)

### Generate the simulation data
If you already have input data, you can skip this section and begin the inference. The example here is to simulate cell dynamics with a doubling time $\tau=20\text{min}$ (growth rate $r=\log 2 /\tau$), and equal division. We choose a sigmoid function for the cell division rate $\lambda(s_t,s_{t}^{\ast}) = \lambda_{\max}[1+\tanh\beta(s_t-\tilde{s}(s_{t}^{\ast}))]/2$, where $\tilde{s}(s_{t}^{\ast}) = s_{t}^{\ast}/2+\Delta$.

In [ ]:
function g_exp(du, u, p, t) # growth rate function g = r*s
    du[1] = r * u[1]
end

function affect!(integrator) # equal division
    integrator.u[2] = integrator.u[1]
    integrator.u[1] = integrator.u[1]/2
end

function rate_tanh_memory(u, p, t) # sigmoidal λ
    a = p[1]
    l0 = p[2]
    w = p[3]
    lam = p[4]
    (1 + tanh((u[1] - (a * l0 + (1 - a) * u[2]))/w)) * lam
end

In [2]:
τ = 1/3 # doubling time 20min
r = log(2)/τ # growth rate

# define parameters outside and pass them to the rate function
ode_params = (0.5, 6.5, 0.8, 20.0) 


# generate the trajectory and save
u0 = [0.5*ode_params[2], ode_params[2]] # (s(0),s^*(0))
T = 5000.0
tspan = (0.0, T)
t = 0:0.01:T
prob = ODEProblem(f_exp, u0, tspan, ode_params)

jump = VariableRateJump(rate_tanh_memory, affect!)
jump_prob = JumpProblem(prob, Direct(), jump)
sol = solve(jump_prob, Tsit5(), dtmax=0.01);

Lsim_memory = Array(sol')[:,1];
Llast_memory = Array(sol')[:,2];
tsim_memory = sol.t;
isdiv_sim_memory, dt_sim_memory = annotate_sim_traj(Lsim_memory, tsim_memory);

writedlm("../figure/slides_data.txt", [Lsim_memory, Llast_memory, tsim_memory], '\t')

2.079441541679836

### Read in the input data

In [10]:
using JSON
data = readdlm("../data/simulated data/simulation_data.txt", '\t')

Lsim_memory = data[1,:];
Llast_memory = data[2,:];
tsim_memory = data[3,:];

isdiv_sim_memory, dt_sim_memory = annotate_sim_traj(Lsim_memory, tsim_memory);


### Correlation

In [ ]:
using StatsBase
hedges = collect(3:0.25:12.5);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h2 = fit(Histogram, (Lsim_memory[isdiv_sim_memory][1:end-1], Lsim_memory[isdiv_sim_memory][2:end]), (hedges,hedges));

h2_normalize = h2.weights / sum(h2.weights) / dhedges^2

svd_h2 = svd(h2_normalize);

p_h2 = plot(; aspect_ratio=1, framestyle=:box, xlim=(3,9), ylim=(3,9), xtick=collect(3:3:10), ytick=collect(3:3:10))
heatmap!( p_h2, hbins, hbins, h2_normalize, clim=(0,0.3),c = :Blues_9,colorbar_ticks=())  

ll=@layout [a b]
plot(p_h2, layout=ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)

savefig("../figure/simulated data/sim_correlation.png")

In [36]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h2.S ./ svd_h2.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=1.5)
savefig("../figure/simulated data/sim_svd.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_svd.pdf"

### Trajectory

In [4]:
p_traj_m = plot(; size=(750,300), xlim=(0,10.05), ylim=(0,13), xtick=collect(0:5:40), ytick=collect(0:5:15), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent) # xlabel="Time", ylabel="Cell length"
plot!(p_traj_m, tsim_memory, Lsim_memory, lw=0.5)
scatter!(tsim_memory[isdiv_sim_memory], Lsim_memory[isdiv_sim_memory], markercolor=:gray, ms=2.2)
savefig("../figure/simulated data/sim_trajectory.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_trajectory.pdf"

### Input data distribution

In [3]:
using StatsBase
hedges = collect(4:0.25:9);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
hh_obs = fit(Histogram, (vcat( [Lsim_memory[i] for i in eachindex(Lsim_memory) ]...), vcat( [Llast_memory[i] for i in eachindex(Llast_memory) ]...) ), (hedges,hedges));

hh_obs_normalize = hh_obs.weights / sum(hh_obs.weights) / dhedges^2

svd_hh_obs = svd(hh_obs_normalize);

ll=@layout [a b]
p_hh_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(4,9), ylim=(4,9), xtick=collect(5:2:10), ytick=collect(5:2:10))#, title="P(i,i-1)" 
heatmap!( p_hh_obs, hbins, hbins, hh_obs_normalize, clim=(0,0.2),c = :Blues_9,
colorbar_ticks=(),colorbar=false) 

plot(p_hh_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=5Plots.mm, background_color=:transparent)
savefig("../figure/simulated data/sim_datadist.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_datadist.png"

### Inference
Here, we construct orthogonal polynomial basis from data using the modified Gram-Schmidt procedure (Please find details in the SI of our paper). The $\lambda$ only incorporates one generation of memory, that is $\lambda(s_t,s_{t}^{\ast})$.

In [77]:
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
n_func1 = [4, 4]
l01 = [5.5, 5.5]
Θ1, Σ1 = mgs_gen_bases( [vcat(Lsim_memory...), vcat(Llast_memory...)], n_func1, bf1, l01);
Θdiv1 = Θ1[vcat(isdiv_sim_memory...),:];
Θdiv1 = sum(Θdiv1, dims=1)[:];

history1 = sparse_optimization_sweep(sparse_optimization_fg, Θ1, Θdiv1, vcat(dt_sim_memory...); init_cutoff=0.0, coeffs0=1e-4, EPSILON=1E-7, MAX_ITERS=20);

===== start of epoch 1 =====
itr=1: -4358.978210392708
itr=2: -4354.337874416371
itr=3: -4351.406763122957
itr=4: -4349.922321710429
itr=5: -4350.404761642095
itr=6: -4348.856703415091
itr=7: -4348.3608583037085
itr=8: -4347.356703413079


┌ Warning: Failed to achieve finite new evaluation point, using alpha=0
└ @ LineSearches /Users/shijiezhang/.julia/packages/LineSearches/G1LRk/src/hagerzhang.jl:148


itr=9: -4347.356703413079


┌ Warning: Failed to achieve finite new evaluation point, using alpha=0
└ @ LineSearches /Users/shijiezhang/.julia/packages/LineSearches/G1LRk/src/hagerzhang.jl:148


itr=1: -4358.978210392708
itr=2: -4354.337874416371
itr=3: -4351.406763122957
itr=4: -4349.922321710429
itr=5: -4350.404761642095
itr=6: -4348.856703415091
itr=7: -4348.3608583037085
itr=8: -4347.356703413079


┌ Warning: Failed to achieve finite new evaluation point, using alpha=0
└ @ LineSearches /Users/shijiezhang/.julia/packages/LineSearches/G1LRk/src/hagerzhang.jl:148


... (7 outputs left out)

┌ Warning: Failed to achieve finite new evaluation point, using alpha=0
└ @ LineSearches /Users/shijiezhang/.julia/packages/LineSearches/G1LRk/src/hagerzhang.jl:148


itr=10: -4347.107194297163


┌ Warning: Failed to achieve finite new evaluation point, using alpha=0
└ @ LineSearches /Users/shijiezhang/.julia/packages/LineSearches/G1LRk/src/hagerzhang.jl:148


itr=1: -4356.429508801301
itr=2: -4353.39895030886
itr=3: -4351.701119846598
itr=4: -4350.532491430369
itr=5: -4349.512371772214
itr=6: -4348.8027152767445
itr=7: -4348.273334815594
itr=8: -4348.041522986858
itr=9: -4348.04148007139
itr=1: -4352.068687814483
itr=2: -4350.042347843639
itr=3: -4349.12656876014
itr=4: -4348.652579430955
itr=5: -4348.298037825154
itr=6: -4347.998889138846
itr=7: -4347.671936210736
itr=8: -4347.368750367596
itr=9: -4347.11971590946
itr=10: -4346.999258210979
itr=11: -4346.966612322537
itr=12: -4346.965684683086
itr=13: -4346.965759118671
itr=1: -4348.88995682576
itr=2: -4347.686285524444
itr=3: -4347.178647190028
itr=4: -4347.032782520671
itr=5: -4346.993719492571
itr=6: -4346.981039712163
itr=7: -4346.980437943455
itr=8: -4346.980055482439
itr=1: -4348.88995682576
itr=2: -4347.686285524444
itr=3: -4347.178647190028
itr=4: -4347.032782520671
itr=5: -4346.993719492571
itr=6: -4346.981039712163
itr=7: -4346.980437943455
itr=8: -4346.980055482439
===== end of 

In [78]:
save_inference_results([vcat(tsim_memory...) vcat(Lsim_memory...) vcat(Llast_memory...)], vcat(isdiv_sim_memory...), vcat(dt_sim_memory...), 
    [ [@sprintf "%s" typeof(bf1[i]) for i in 1:length(bf1)] n_func1 l01 ], history1; 
    data_type = "simulationdata", learn_type = "learn_memory1", output_dir = "./output/", 
    use_uid=true, save_traj=true
)

### Read in and analyze the inference result

In [6]:
filepath1 = "./output/simulationdata_learn_memory1/24-05-14-T1620/"

function rebuild_library_params(filepath)
    library_sum = readdlm(filepath * "library");
    bf_string = library_sum[:, 1];
    @show bf_string
    n_func = convert(Vector{Int}, library_sum[:,2]);
    l0 = convert(Vector{Float64}, library_sum[:,3]);
    return n_func, l0
end

n_func1, l01 = rebuild_library_params(filepath1);
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
_, Σ1 = mgs_gen_bases(Ls1, n_func1, bf1, l01);

traj = readdlm(filepath1 * "traj");
tobs = traj[:,1];
Ls1 = [traj[:,j] for j in 2:3];

dtobs = readdlm(filepath1 * "dts")[:];
isdiv = convert( BitVector, readdlm(filepath1 * "isdiv")[:] );

bf_string = Any["sChebyshevT{Float64}", "sChebyshevT{Float64}"]


In [7]:
function rebuild_history(filepath)

    history_coeffs = readdlm(filepath * "history_coeffs");
    cache_summary  = readdlm(filepath * "history_summary");
    keynames = convert(Vector{String}, cache_summary[1,:] );
    history_metric = cache_summary[2:end,:];
    history = Dict(zip(Symbol.(keynames), [history_metric[:,i] for i = 1:length(keynames)]));

    return history, history_coeffs

end

history1, history_coeffs1 = rebuild_history(filepath1);

To select a parsimonious model that best explain the data with the least number of terms, we employ a modified (negative) BIC score $ \sim  \ln P(s_t | \hat{\mathbf{w}} ) - 1/2\ln |\hat{\mathbf{H}}|$ for stochastic models with non-Gaussian noise. Here, $\hat{\mathbf{H}}$ denotes the Hessian of the log posterior evaluated at $\hat{\mathbf{w}}$ and its log-determinant $\ln |\hat{\mathbf{H}}|$ serves as a penalty term for the number of model parameters.

In [ ]:
_, imax1 = findmax( -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs) )
BIC = -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs)
plot(history1[:nterms], BIC, lw=2, lc=:dodgerblue4,
    xtick=[0,10,20],ytick=[0,2,4],ylim=(-1.1,4.5), size=(330,300), bottommargin=-6Plots.mm, leftmargin=0Plots.mm,
    background_color=:transparent)
scatter!([history1[:nterms][imax1]], [BIC[imax1]], ms=2.5, mc=:dodgerblue4, shape=:rect)
savefig("../figure/simulated data/sim_BIC.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_BIC.pdf"

In [9]:
heatmap(reshape(history_coeffs1[:, imax1],5,5), c =:broc, ticks=false, axis=true,aspect_ratio =1, clim=(-1,1),
xlim=(0.5,5.5), ylim=(0.5,5.5),framestyle=:box,size=(300,300), background_color=:transparent, yflip = true
)
savefig("../figure/simulated data/sim_coeffs_matrix.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_coeffs_matrix.png"

In [10]:
ll = collect(4:0.1:9);
ll_ = collect(4:0.1:9);
ln_λ_inf = mgs_rec_bases([ll, ll_], Σ1, history_coeffs1[:, imax1], bf1, l01);

pl1 = plot(;size=(300,300), bottommargin=-6Plots.mm, leftmargin=-3Plots.mm, background_color=:transparent);
heatmap!(pl1, ll, ll_, exp.(ln_λ_inf'), clim=(0,22), 
xlim=(ll[1],ll[end]), ylim=(ll_[1],ll_[end]), aspect_ratio=1, framestyle=:box, 
c=:lapaz, xtick=[5,7,9],ytick=[5,7,9])
plot!(pl1, ll, (ll /0.5 .- 6.5), lc=:black, ls=:dash, lw=1.0)

savefig("../figure/simulated data/sim_lam.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_lam.png"

### Growth function $g$ inference
We assume the cell growth with a linear function $g(s_{t}) = g_0 + g_1 s_{t}$ and use linear regression t infer the parameter $g_0$ and $g_1$.

We first use interpolation to smooth the growth function $g$.

In [11]:
function interpolation(t, L, deg)
    A = ones(length(L), deg);
    for i = 2:deg
        A[:,i] = t.^(i-1);
    end
    coeff = A\L;
    D = zeros(length(L), deg);
    for i = 2:deg
        D[:,i] = (i-1) * t.^(i-2);
    end
    return coeff, D * coeff, A * coeff
end

interpolation (generic function with 1 method)

In [12]:
bf_g = Polynomial()
Y_g = (Lsim_memory[2:end] .- Lsim_memory[1:end-1]) ./ (tsim_memory[2:end] .- tsim_memory[1:end-1]);
Lt_g = Lsim_memory[1:end-1];
Lt_g = Lt_g[Y_g .> 0.0]; 
Y_g = Y_g[Y_g .> 0.0];
Theta_g = gen_bases( Lt_g,  2);

COEFF_TRESHOLD = 0.02
_, coeffs_g = sparse_bayesian_optimization(Theta_g, Y_g)
coeffs_g[abs.(coeffs_g) .< COEFF_TRESHOLD] .= 0.0

1-element view(::Vector{Float64}, [3]) with eltype Float64:
 0.0

The red line represents our inferred result and the gray line represents the ground truth.

In [15]:
histogram2d(Lt_g, Y_g, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, 
xlim = (2,8), ylim = (-10,30), xtick =[2, 5, 8], ytick=[0, 15, 30], bins=(2:0.4:8, -10:2:30), c=cgrad(:tempo, rev=false),clim=(0,0.2)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,
plot!([2,8], r*[2,8], lw=3.0, lc=:gray)
plot!([2,8], coeffs_g[1] .+ coeffs_g[2]*[2,8], lw=1.0, lc=:red, ls=:dash)
savefig("../figure/simulated data/sim_inferred_g.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_inferred_g.png"

### Cut size function $h$ inference
Similar to growth function $g$, we assume the cut size of a cell at division is also modeled by a linear function $h(s_{t^{-}}) = h_0 + h_1 s_{t^{-}}$. 

In [16]:
bf_h = Polynomial()
Y_h =  Lsim_memory[1:end-1][isdiv_sim_memory[1:end-1]] .- Lsim_memory[2:end][isdiv_sim_memory[1:end-1]]
Theta_h = gen_bases( Lsim_memory[1:end-1][isdiv_sim_memory[1:end-1]], 2);

coeffs_h = Theta_h\ Y_h
coeffs_h[abs.(coeffs_h) .< 0.02] .= 0.0

2-element view(::Vector{Float64}, [1, 3]) with eltype Float64:
 0.0
 0.0

The red line represents our inferred result and the gray line represents the ground truth.

In [19]:
histogram2d(Lsim_memory[isdiv_sim_memory], Y_h, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-2Plots.mm,
xlim = (3,9), ylim = (0,6), xtick =[3, 6, 9], ytick=[0, 3, 6], bins=(3:0.2:9, 0:0.2:6), c=cgrad(:tempo, rev=false),clim=(0,2)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,)
plot!([0, 20], 0.5*[0, 20], lw=3.0, lc=:gray)
plot!([0, 20], coeffs_h[1] .+ coeffs_h[2] * [0, 20].^1, label="learned", lw=1.0, lc=:red, ls=:dash)
savefig("../figure/simulated data/sim_inferred_h.png")



"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_inferred_h.png"

### Resimulation based on inferred results
Next, we resimulate a trajectory and compare it with the input data.

In [20]:
function f(du, u, p, t)
    bf_g = p[1]
    coeffs_g = p[2]
    du[1] = 0.0
    for n = 1:length(coeffs_g)
        du[1] += coeffs_g[n]*bf_g(n-1, u[1])
    end
end

function rate_memory(u, p, t)
    Σ1 = p[5]
    coeffs = p[6]
    bf1 = p[7]
    l01 = p[8]
    ln_rate = mgs_rec_bases([[u[1]], [u[2]]], Σ1, coeffs, bf1, l01);
    return exp(ln_rate[1])
end

function affect!(integrator)
    integrator.u[2] = integrator.u[1]
    bf_h = integrator.p[3]
    coeffs_h = integrator.p[4]
    u_old = integrator.u[1]
    for n = 1:length(coeffs_h)
        integrator.u[1] -= coeffs_h[n] * bf_g(n-1, u_old)
    end
end

affect! (generic function with 1 method)

In [108]:
# resimulate data and save or read in your resimulated data
# resimulate data based on your inferred parameters
u0 = [0.5*6.5, 6.5]

vrj = VariableRateJump(rate_memory, affect!)
oprob = ODEProblem(f, u0, (0.0,maximum(tsim_memory)*5), (bf_g, coeffs_g, bf_h, coeffs_h, Σ1, history_coeffs1[:, imax1], bf1, l01))
jprob = JumpProblem(oprob, Direct(), vrj)
sol = solve(jprob,Tsit5(),dtmax=0.01);
Lresult = Array(sol')[:,1];
tresult = sol.t;

isdiv_learn, dt_learn, Llast_resim, Llast2_memory = annotate_exp_traj3(Lresult, tresult);

writedlm("sim_resim_data.txt", [Lresult, Llast_resim, tresult], '\t')

┌ Warning: Instability detected. Aborting
└ @ SciMLBase /Users/shijiezhang/.julia/packages/SciMLBase/2HZ5m/src/integrator_interface.jl:602


In [8]:
# read in resimulated data
using JSON
resim_data = readdlm("../figure/simulated data/sim_resim_data.txt", '\t')

Lresult = resim_data[1, :];
Llast_resim = resim_data[2, :];
tresult = resim_data[3, :];

isdiv_result = []
isdiv_learn, dt_learn = annotate_sim_traj(Lresult, tresult);

In [26]:
p_traj_m = plot(; size=(750,300), xlim=(0,10), ylim=(0,13), xtick=collect(0:5:40), ytick=collect(0:5:15), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent) # xlabel="Time", ylabel="Cell length"
plot!(p_traj_m, tresult, Lresult, lw = 1.5, lc=:seagreen)
scatter!(tresult[isdiv_learn], Lresult[isdiv_learn], markercolor=:seagreen, ms=2.2)
savefig("../figure/simulated data/sim_resim_trajectory.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_resim_trajectory.pdf"

In [ ]:
using StatsBase
hedges = collect(3:0.25:10.5);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h2_ = fit(Histogram, (Lresult[isdiv_learn][1:end-1], Lresult[isdiv_learn][2:end]), (hedges,hedges));

h2_normalize_ = h2_.weights/sum(h2_.weights) / dhedges^2

svd_h2_ = svd(h2_normalize_);

p_h2_ = plot(; aspect_ratio=1, framestyle=:box, xlim=(3,9), ylim=(3,9), xtick=collect(3:3:10), ytick=collect(3:3:10))
heatmap!( p_h2_, hbins, hbins, h2_normalize_, clim=(0,0.3),c = :Blues_9,colorbar_ticks=()) 

ll=@layout [a b]
plot(p_h2_, layout=ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)

savefig("../figure/simulated data/sim_resim_correlation.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_resim_correlation.png"

In [37]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h2.S ./ svd_h2.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=1.5)
scatter!( p_S_result, svd_h2_.S ./ svd_h2_.S[1], markercolor=:seagreen, ms=2, markershape=:circle,msw=1, msc=:seagreen)
savefig("../figure/simulated data/sim_svd_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/simulated data/sim_svd_resim.pdf"